In [0]:
%run ./00_config

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.client_360 AS
WITH acc AS (                                                   -- sub-total 1: balances per client
    SELECT client_id,
           SUM(CASE WHEN product IN ('Private Banking Account', 'Signature Account') THEN balance END) AS main_balance_zar,
           SUM(CASE WHEN product = 'Tax-Free Savings' THEN balance END)  AS tfsa_balance_zar,
           SUM(CASE WHEN product = 'Optimum Offshore' THEN balance END)  AS offshore_balance_usd,  -- offshore is in USD
           COUNT(*) AS account_count
    FROM {SILVER}.accounts
    GROUP BY client_id
),
crd AS (                                                        -- sub-total 2: credit cards per client
    SELECT a.client_id,
           SUM(c.credit_limit_zar)    AS credit_limit_zar,
           SUM(c.current_balance_zar) AS card_balance_zar       -- NEW: what the client owes on cards today
    FROM {SILVER}.cards c
    JOIN {SILVER}.accounts a ON a.account_id = c.account_id     -- cards link to clients via their account
    GROUP BY a.client_id
),
lns AS (                                                        -- sub-total 3: loans per client
    SELECT client_id,
           COUNT(*)              AS loan_count,
           SUM(outstanding_zar)  AS loans_outstanding_zar,
           MAX(loan_type = 'Home loan')       AS has_home_loan,
           MAX(loan_type = 'Vehicle finance') AS has_vehicle_finance
    FROM {SILVER}.loans
    GROUP BY client_id
),
txn AS (                                                        -- sub-total 4: recent activity per client
    SELECT a.client_id,
           MAX(t.txn_date_sast) AS last_txn_date,
           SUM(CASE WHEN t.direction = 'debit' AND t.txn_date_sast > date_sub(DATE'{RUN_DATE}', 90)
                    THEN -t.amount_zar ELSE 0 END) AS spend_last_90d_zar,         -- money out, shown positive
           COUNT_IF(t.txn_date_sast > date_sub(DATE'{RUN_DATE}', 90)) AS txns_last_90d
    FROM {SILVER}.transactions t
    JOIN {SILVER}.accounts a ON a.account_id = t.account_id
    GROUP BY a.client_id
)
SELECT c.client_id, c.first_name, c.last_name, c.segment, c.risk_profile, c.city, c.province,
       FLOOR(months_between(DATE'{RUN_DATE}', c.date_of_birth) / 12) AS age,
       FLOOR(months_between(DATE'{RUN_DATE}', c.client_since) / 12)  AS years_as_client,
       c.monthly_income_zar,
       c.rm_id, r.rm_name, r.branch,
       COALESCE(acc.main_balance_zar, 0)     AS main_balance_zar,
       COALESCE(acc.tfsa_balance_zar, 0)     AS tfsa_balance_zar,
       COALESCE(acc.offshore_balance_usd, 0) AS offshore_balance_usd,
       acc.account_count,
       COALESCE(crd.credit_limit_zar, 0)     AS credit_limit_zar,
       COALESCE(crd.card_balance_zar, 0)     AS card_balance_zar,
       ROUND(100 * crd.card_balance_zar / NULLIF(crd.credit_limit_zar, 0), 1) AS card_utilisation_pct,
       COALESCE(lns.loan_count, 0)            AS loan_count,
       COALESCE(lns.loans_outstanding_zar, 0) AS loans_outstanding_zar,
       COALESCE(lns.has_home_loan, false)     AS has_home_loan,
       COALESCE(lns.has_vehicle_finance, false) AS has_vehicle_finance,
       ROUND(COALESCE(lns.loans_outstanding_zar, 0) / (c.monthly_income_zar * 12), 2) AS debt_to_annual_income,
       txn.last_txn_date,
       COALESCE(txn.spend_last_90d_zar, 0) AS spend_last_90d_zar,
       COALESCE(txn.txns_last_90d, 0)      AS txns_last_90d,
       current_timestamp() AS _gold_built_at
FROM {SILVER}.clients c                                          -- start from every client...
LEFT JOIN {SILVER}.relationship_managers r ON r.rm_id = c.rm_id  -- ...and attach the rest, keeping clients with no match
LEFT JOIN acc ON acc.client_id = c.client_id
LEFT JOIN crd ON crd.client_id = c.client_id
LEFT JOIN lns ON lns.client_id = c.client_id
LEFT JOIN txn ON txn.client_id = c.client_id
""")

print("client_360:", spark.table(f"{GOLD}.client_360").count())
display(spark.table(f"{GOLD}.client_360").limit(5))

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.anomaly_transactions AS
SELECT a.client_id, t.txn_id, t.txn_ts, t.txn_date_sast, t.txn_hour_sast,
       t.amount_zar, t.merchant_name, t.country_code, t.channel,
       current_timestamp() AS _gold_built_at
FROM {SILVER}.transactions t
JOIN {SILVER}.accounts a ON a.account_id = t.account_id   -- to know whose transaction it is
WHERE t.channel = 'Online'                                -- online purchase
  AND t.country_code <> 'ZA'                              -- foreign merchant
  AND t.amount_zar <= -40000                              -- R40,000 or more going out
  AND t.txn_hour_sast < 6                                 -- between 00:00 and 05:59 SAST
""")
print("anomaly_transactions:", spark.table(f"{GOLD}.anomaly_transactions").count())

In [0]:
# Forex windows copied from the Lab 2 hunt so results match exactly.
# Tech debt: derive these from RUN_DATE later.
FOREX_FIRST_END = "2026-01-07"   # first 3 months of history end here
FOREX_LAST_START = "2026-07-09"  # last 3 months start here

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.client_signals AS
WITH idle AS (                                     -- idle_cash.sql
    SELECT c.client_id, ROUND(a.balance / c.monthly_income_zar, 1) AS idle_months_of_income
    FROM {SILVER}.clients c
    JOIN {SILVER}.accounts a ON a.client_id = c.client_id
    WHERE a.product IN ('Private Banking Account', 'Signature Account')
      AND a.balance > 4 * c.monthly_income_zar
),
forex AS (                                         -- forex_growth.sql
    SELECT a.client_id,
           COUNT_IF(t.txn_date_sast <  DATE'{FOREX_FIRST_END}')  AS forex_trips_first_3m,
           COUNT_IF(t.txn_date_sast >= DATE'{FOREX_LAST_START}') AS forex_trips_last_3m
    FROM {SILVER}.transactions t
    JOIN {SILVER}.accounts a ON a.account_id = t.account_id
    WHERE t.merchant_category = 'travel_foreign'
    GROUP BY a.client_id
    HAVING forex_trips_last_3m >= 8
),
upgrade AS (                                       -- upgrade.sql
    SELECT client_id
    FROM {SILVER}.clients
    WHERE segment = 'Private Banking' AND monthly_income_zar >= 92000
),
salary AS (                                        -- salary_increase.sql
    SELECT a.client_id,
           MIN(t.amount_zar) AS lowest_salary_zar,
           MAX(t.amount_zar) AS highest_salary_zar,
           ROUND((MAX(t.amount_zar) / MIN(t.amount_zar) - 1) * 100, 1) AS salary_raise_pct
    FROM {SILVER}.transactions t
    JOIN {SILVER}.accounts a ON a.account_id = t.account_id
    WHERE t.merchant_category = 'income'
    GROUP BY a.client_id
    HAVING MAX(t.amount_zar) > MIN(t.amount_zar)
),
anom AS (                                          -- anomaly, rolled up per client
    SELECT client_id, COUNT(*) AS anomaly_txn_count, MIN(amount_zar) AS anomaly_largest_zar
    FROM {GOLD}.anomaly_transactions
    GROUP BY client_id
)
SELECT c.client_id,
       idle.client_id IS NOT NULL    AS idle_cash,
       idle.idle_months_of_income,
       forex.client_id IS NOT NULL   AS forex_growth,
       forex.forex_trips_first_3m, forex.forex_trips_last_3m,
       upgrade.client_id IS NOT NULL AS upgrade_candidate,
       salary.client_id IS NOT NULL  AS salary_increase,
       salary.lowest_salary_zar, salary.highest_salary_zar, salary.salary_raise_pct,
       anom.client_id IS NOT NULL    AS anomaly,
       COALESCE(anom.anomaly_txn_count, 0) AS anomaly_txn_count,
       anom.anomaly_largest_zar,
       CAST(idle.client_id IS NOT NULL AS INT) + CAST(forex.client_id IS NOT NULL AS INT)
         + CAST(upgrade.client_id IS NOT NULL AS INT) + CAST(salary.client_id IS NOT NULL AS INT)
         + CAST(anom.client_id IS NOT NULL AS INT) AS signal_count,   -- how many reasons to call
       current_timestamp() AS _gold_built_at
FROM {SILVER}.clients c                            -- every client gets a row, even with no signals
LEFT JOIN idle    ON idle.client_id    = c.client_id
LEFT JOIN forex   ON forex.client_id   = c.client_id
LEFT JOIN upgrade ON upgrade.client_id = c.client_id
LEFT JOIN salary  ON salary.client_id  = c.client_id
LEFT JOIN anom    ON anom.client_id    = c.client_id
""")
print("client_signals:", spark.table(f"{GOLD}.client_signals").count())

In [0]:
display(spark.sql(f"""
SELECT COUNT_IF(idle_cash) AS idle_cash, COUNT_IF(forex_growth) AS forex_growth,
       COUNT_IF(upgrade_candidate) AS upgrade_candidate, COUNT_IF(salary_increase) AS salary_increase,
       COUNT_IF(anomaly) AS anomaly_clients, SUM(anomaly_txn_count) AS anomaly_txns,
       COUNT_IF(signal_count = 0) AS no_signals
FROM {GOLD}.client_signals
"""))

In [0]:
import os

os.makedirs(f"{EXPORT_PATH}/answers", exist_ok=True)   # folder for the hunt answers
os.makedirs(f"{EXPORT_PATH}/gold", exist_ok=True)      # folder for the Gold tables

sig = spark.table(f"{GOLD}.client_signals")
for story in ["idle_cash", "forex_growth", "upgrade_candidate", "salary_increase"]:
    (sig.filter(story).select("client_id")             # keep clients whose flag is true
        .toPandas().to_csv(f"{EXPORT_PATH}/answers/{story}.csv", index=False))

(spark.table(f"{GOLD}.anomaly_transactions").select("txn_id")   # anomaly is scored per transaction
    .toPandas().to_csv(f"{EXPORT_PATH}/answers/anomaly.csv", index=False))

for t in ["client_360", "client_signals", "anomaly_transactions"]:
    (spark.table(f"{GOLD}.{t}").drop("_gold_built_at")
        .toPandas().to_csv(f"{EXPORT_PATH}/gold/{t}.csv", index=False))

display(dbutils.fs.ls(f"{EXPORT_PATH}/answers"))
display(dbutils.fs.ls(f"{EXPORT_PATH}/gold"))